# Coqui TTS

A refresher on **Coqui TTS** — the open-source Python toolkit (`coqui-tts` / the `TTS` package) for *text-to-speech*: training and running models that turn a string into a waveform. It bundles a model zoo (Tacotron2, Glow-TTS, VITS, YourTTS, **XTTS-v2**), a unified `TTS` API, and a CLI. Coqui the company wound down in early 2024, but the library lives on as a community-maintained fork — it's still the most batteries-included open TTS stack, and the one to know for **zero-shot voice cloning** (XTTS-v2).

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _the text front-end and a Griffin-Lim vocoder run on CPU with no download; real model loads are gated behind `RUN_COQUI`_

## 1. What & Why

**What it is.** Coqui TTS is a PyTorch toolkit that wraps the whole text-to-speech pipeline behind one API. `from TTS.api import TTS` gives you `list_models()`, `tts()`, and `tts_to_file()`; the CLI gives you `tts --text "..." --out_path out.wav`. Under the hood it ships dozens of pretrained checkpoints covering the major architecture families — autoregressive (Tacotron2), flow-based (Glow-TTS), end-to-end adversarial (VITS), and large multilingual voice-cloning models (**XTTS-v2**, YourTTS, plus wrappers for Bark and Tortoise).

**The problem it solves.** Before Coqui, open TTS meant stitching together a research repo for the acoustic model, a *separate* repo for the vocoder, your own text-normalization/phonemizer front end, and a pile of incompatible config formats. Coqui (descended from Mozilla TTS) unified all of that: one config schema, one trainer, one inference API, and a model zoo where acoustic models and vocoders are already paired and downloadable by name. It made "synthesize speech in Python in three lines" real, and made *training your own voice* tractable.

**When to reach for it.** Offline/self-hosted TTS where you don't want a per-character cloud bill or your audio leaving the box; **voice cloning** from a few seconds of reference audio (XTTS-v2 is the headline feature); multilingual synthesis (~17 languages in XTTS); fine-tuning or training a custom voice on your own dataset; research/experimentation across TTS architectures with a common harness.

**When not to.** If you need the absolute lowest-latency, lightest on-device synthesis, **Piper** is leaner. If you want zero-setup top-tier quality and will pay for it, **ElevenLabs / Azure / Google** cloud TTS beat it on naturalness and ops. And the project is now community-maintained (no company behind it), so for a long-lived production commitment, weigh that against a supported vendor.

## 2. Mental Model

**Coqui is an assembly line with two stations: an *acoustic model* that turns text into a mel-spectrogram (a picture of the sound), and a *vocoder* that turns that picture into an actual waveform. The `TTS` API just bolts a matched pair together and hides the seam.**

```
  text          FRONT END                 ACOUSTIC MODEL              VOCODER
 ┌───────┐  normalize + phonemize   ┌────────────────────┐   ┌──────────────────────┐
 │"Hello"│ ─▶ token ids ──────────▶ │ Tacotron2 / Glow / │ ─▶│ HiFi-GAN / WaveGRAD / │ ─▶ 22 kHz wav
 └───────┘   (chars or phonemes)    │ VITS encoder       │   │ (or Griffin-Lim)      │
                                    │ -> mel spectrogram │   │ mel -> waveform       │
                                    └────────────────────┘   └──────────────────────┘
                                            \___________ VITS / XTTS fuse both stages (end-to-end) ___________/
```

Three ideas make everything click:

1. **Two stages, decoupled.** The classic models (Tacotron2, Glow-TTS) emit a **mel-spectrogram** and hand it to a *separate* neural **vocoder** (HiFi-GAN, etc.). That's why model names in the zoo often come as a pair, and why you can mix-and-match. **VITS and XTTS are end-to-end** — they fold both stages into one network and output a waveform directly.
2. **The model name *is* the config.** Checkpoints are addressed like a path: `tts_models/<lang>/<dataset>/<model>`, e.g. `tts_models/en/ljspeech/vits`. That string selects architecture, training data, sample rate, and the paired vocoder.
3. **Cloning = conditioning, not training.** XTTS-v2 doesn't fine-tune to clone a voice; it *conditions* on a speaker embedding extracted from a few seconds of reference audio at inference time. No training step, no per-voice checkpoint.

## 3. Key Concepts

- **Front end (text → tokens).** Text normalization (expand numbers, currency, abbreviations: `Dr.`→`doctor`, `$5`→`five dollars`) followed by tokenization into **characters or phonemes**. Coqui can phonemize via `espeak-ng`/`gruut` (IPA tokens) or run on raw characters. Garbage in (un-normalized text) → garbage out (mispronounced audio).
- **Acoustic model (tokens → mel).** Predicts an `(n_mels × frames)` mel-spectrogram. Families: **Tacotron2** (autoregressive, attention-based, can mis-attend on long text), **Glow-TTS** (flow-based, non-autoregressive, robust durations), **FastPitch/FastSpeech**-style (fast, controllable pitch/duration).
- **Vocoder (mel → waveform).** Converts the spectrogram to audio. **HiFi-GAN** is the quality/speed default; WaveGRAD/WaveRNN are alternatives; **Griffin-Lim** is the classical, training-free baseline (recovers phase iteratively from magnitude — fast but buzzy). Neural vocoders exist precisely because Griffin-Lim's guessed phase sounds robotic.
- **End-to-end models.** **VITS** combines a flow-based acoustic model, a variational autoencoder, and an adversarial vocoder into one network trained jointly — no separate vocoder. **XTTS-v2** extends this with a GPT-style token model for **multilingual, zero-shot voice cloning**.
- **Speaker / language conditioning.** Multi-speaker models take a `speaker` (id or embedding) and multilingual ones a `language`. XTTS clones by passing `speaker_wav=<reference clip>` — a 6–20 s sample is enough.
- **The model zoo & naming.** `TTS().list_models()` enumerates everything; ids look like `tts_models/en/ljspeech/tacotron2-DDC` or `tts_models/multilingual/multi-dataset/xtts_v2`. Vocoders live under `vocoder_models/...`.
- **Sample rate.** Most Coqui checkpoints are **22.05 kHz** (XTTS outputs 24 kHz). Mixing rates between stages, or against your downstream pipeline, produces chipmunk/slow-mo audio.

## 4. Setup

```bash
# Community-maintained fork (use this — the original `TTS` PyPI name is unmaintained):
pip install coqui-tts            # imports as `TTS`; pulls in torch, torchaudio, etc.

# Phonemizer back end for IPA-based models (optional but recommended):
#   apt-get install espeak-ng        # Debian/Ubuntu
#   brew install espeak-ng           # macOS
```

First synthesis downloads the chosen checkpoint from the model zoo and caches it under
`~/.local/share/tts` (Linux) / `~/Library/Application Support/tts` (macOS). Sizes vary a
lot: a single-speaker VITS is ~100–150 MB; **XTTS-v2 is ~1.8 GB** plus it requires
accepting the Coqui Public Model License (set `COQUI_TOS_AGREED=1` to auto-agree in code).

To stay self-contained, the runnable cells below reproduce Coqui's **text front end** and
a **Griffin-Lim vocoder** with only `numpy` + `torch` — **no download, CPU-only**. The real
`TTS.api` calls (VITS synthesis and XTTS cloning) are shown but gated behind `RUN_COQUI`,
so the notebook always executes.

In [ ]:
import sys
import numpy as np
import torch

print(f"python {sys.version.split()[0]}, numpy {np.__version__}, torch {torch.__version__}")
print("Coqui pipeline = front end (text->tokens) -> acoustic model (tokens->mel) -> vocoder (mel->wav)")
print("The next two cells reproduce stages 1 and 3 with no model download.")

## 5. Worked Examples

### Example 1 — The text front end: normalize, then tokenize

Every TTS model consumes **token ids**, not raw text. Coqui's front end first *normalizes*
(lowercase, expand numbers/currency/abbreviations) and then maps each symbol to an id from
a fixed vocabulary (characters here; phonemes if you enable `espeak-ng`). Skipping
normalization is the #1 cause of "it said 'dollar sign five' instead of 'five dollars'".
We reproduce that contract with pure Python — deterministic, no dependencies.

In [ ]:
import re

ABBREV = {"dr.": "doctor", "mr.": "mister", "mrs.": "missus", "st.": "saint"}
DIGITS = {"0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
          "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine"}

def say_number(n):                                     # toy: read digits one by one
    return " ".join(DIGITS[d] for d in n)

def normalize(text):
    text = text.lower().strip()
    for k, v in ABBREV.items():
        text = text.replace(k, v)
    text = re.sub(r"\$(\d+)", lambda m: say_number(m.group(1)) + " dollars", text)
    text = re.sub(r"\d+", lambda m: say_number(m.group(0)), text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# A Coqui-style symbol set; the LIST ORDER is the model's vocabulary (id = index).
PAD, BLANK, EOS = "<pad>", "<blk>", "<eos>"
punct = list(" ,.!?'-")
letters = list("abcdefghijklmnopqrstuvwxyz")
symbols = [PAD, BLANK, EOS] + punct + letters
stoi = {s: i for i, s in enumerate(symbols)}

def encode(text):
    ids = [stoi[c] for c in normalize(text) if c in stoi]
    return ids + [stoi[EOS]]

raw = "Dr. Smith paid $5, OK?"
ids = encode(raw)
print(f"raw       : {raw!r}")
print(f"normalized: {normalize(raw)!r}")
print(f"vocab size: {len(symbols)} symbols (pad/blank/eos + punctuation + a-z)")
print(f"token ids : {ids}")
print(f"round-trip: {''.join(symbols[i] for i in ids)}")

### Example 2 — The vocoder stage: mel/spectrogram → waveform via Griffin-Lim

The acoustic model's output is a **spectrogram** — a magnitude-only picture of the sound
with no phase. A vocoder's whole job is to turn that back into a waveform. Coqui's default
neural vocoder is HiFi-GAN, but the classical, *training-free* baseline is **Griffin-Lim**:
it iteratively guesses the missing phase. We run it on a synthetic tone (no model needed) to
make the stage concrete — and to show *why* neural vocoders exist (Griffin-Lim's guessed
phase leaves audible error).

In [ ]:
SR, N_FFT, HOP = 22050, 1024, 256          # Coqui's common 22.05 kHz config

# Stand in for an acoustic model's output: a 0.5 s vowel-ish tone (3 harmonics).
dur = 0.5
t = np.linspace(0, dur, int(SR * dur), endpoint=False)
sig = (0.6*np.sin(2*np.pi*220*t) + 0.3*np.sin(2*np.pi*440*t)
       + 0.1*np.sin(2*np.pi*880*t)).astype(np.float32)
sig *= np.hanning(sig.size).astype(np.float32)        # fade in/out

wav = torch.from_numpy(sig)
win = torch.hann_window(N_FFT)
mag = torch.stft(wav, N_FFT, HOP, window=win, return_complex=True).abs()  # magnitude only
print(f"spectrogram handed to the vocoder: {tuple(mag.shape)}  (freq_bins, frames) - no phase")

def griffin_lim(mag, n_iter=32):
    """Recover a waveform from magnitude alone by iteratively estimating phase."""
    spec = mag * torch.exp(2j * np.pi * torch.rand(mag.shape))   # random initial phase
    for _ in range(n_iter):
        wav = torch.istft(spec, N_FFT, HOP, window=win)
        est = torch.stft(wav, N_FFT, HOP, window=win, return_complex=True)
        spec = mag * (est / est.abs().clamp(min=1e-8))           # keep mag, take new phase
    return torch.istft(spec, N_FFT, HOP, window=win)

recon = griffin_lim(mag)
n = min(recon.numel(), wav.numel())
err = (recon[:n] - wav[:n]).abs().mean().item()
print(f"reconstructed waveform: {recon.numel()} samples = {recon.numel()/SR:.2f}s @ {SR} Hz")
print(f"mean abs error vs original: {err:.3f}  -> phase guessing is lossy; HiFi-GAN beats this")

### Example 3 — The real `TTS.api`: VITS synthesis and XTTS-v2 cloning (gated)

With `coqui-tts` installed, synthesis is a few lines. The model download is large
(~150 MB for VITS, ~1.8 GB for XTTS-v2), so this is gated behind `RUN_COQUI=1`. Either way
the cell prints the canonical call shapes — single-speaker synth, multilingual zero-shot
**voice cloning** with `speaker_wav`, and the equivalent CLI.

In [ ]:
import os

if os.getenv("RUN_COQUI"):
    from TTS.api import TTS
    # Single-speaker, end-to-end VITS (no separate vocoder needed).
    tts = TTS("tts_models/en/ljspeech/vits")
    tts.tts_to_file(text="Coqui turns text into speech.", file_path="out.wav")
    print("wrote out.wav")

    # Zero-shot voice cloning + multilingual with XTTS-v2 (needs a reference clip).
    os.environ["COQUI_TOS_AGREED"] = "1"
    xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
    xtts.tts_to_file(text="Now in a cloned voice.", file_path="clone.wav",
                     speaker_wav="reference.wav", language="en")
    print("wrote clone.wav")
else:
    print("Set RUN_COQUI=1 (and `pip install coqui-tts`) to synthesize for real.\n")
    print("# list everything in the zoo")
    print("from TTS.api import TTS")
    print("print(TTS().list_models())          # tts_models/... and vocoder_models/...\n")
    print("# single-speaker synthesis (VITS, end-to-end)")
    print('tts = TTS("tts_models/en/ljspeech/vits")')
    print('tts.tts_to_file(text="Hello there.", file_path="out.wav")\n')
    print("# zero-shot voice cloning + multilingual (XTTS-v2, ~1.8 GB)")
    print('xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")')
    print('xtts.tts_to_file(text="Bonjour.", file_path="clone.wav",')
    print('                 speaker_wav="reference.wav", language="fr")\n')
    print("# equivalent CLI")
    print('tts --text "Hello" --model_name "tts_models/en/ljspeech/vits" --out_path out.wav')

## 6. Gotchas & Pitfalls

- **Install the right package.** `pip install TTS` grabs the *original, unmaintained* PyPI
  name; use **`pip install coqui-tts`** (the community fork). Both import as `TTS`, which
  causes endless confusion — check the version, not the import.
- **XTTS license gate.** Loading `xtts_v2` prompts you to accept the Coqui Public Model
  License (CPML, non-commercial-ish terms). In non-interactive code it hangs waiting for
  input — set `os.environ["COQUI_TOS_AGREED"] = "1"` (and actually read the license).
- **Sample-rate mismatches.** Checkpoints are mostly 22.05 kHz; XTTS outputs 24 kHz. If you
  resample wrong or feed a vocoder a mel built at a different rate, you get chipmunk or
  slow-motion audio. Always check the model's `output_sample_rate`.
- **Text normalization is on you for edge cases.** Numbers, currency, dates, URLs, and
  abbreviations need expanding before synthesis. Un-normalized input is the top cause of
  mispronunciation — normalize first (Example 1).
- **Tacotron2 attention failures.** Autoregressive models can babble, skip, or loop on
  long or unusual text. Split into sentences, or prefer a non-autoregressive model
  (Glow-TTS / VITS) for robustness.
- **`espeak-ng` must be installed for phoneme models.** Phonemizer-based checkpoints fail
  at runtime if the `espeak-ng` system binary is missing — install it via your OS package
  manager, not pip.
- **CPU is slow for XTTS.** XTTS-v2 is a ~big GPT-style model; on CPU a sentence can take
  many seconds. Use a GPU (`TTS(...).to("cuda")`) for anything interactive.
- **Cloning quality depends on the reference clip.** Give XTTS 6–20 s of *clean, single-
  speaker* audio at a good sample rate. Noisy or multi-speaker references clone the noise.
- **Maintenance status.** Coqui (the company) shut down in early 2024; the library is
  community-maintained. Pin versions and don't assume long-term commercial support.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs Coqui |
|---|---|---|
| **Coqui TTS (XTTS-v2 / VITS)** | Self-hosted, voice cloning, multilingual, training your own voice | Heavy deps, GPU-hungry for XTTS, community-maintained (no vendor) |
| **Piper** | Fast, tiny, on-device/embedded TTS (Raspberry Pi, Home Assistant) | Lower naturalness; no zero-shot cloning; fewer voices |
| **StyleTTS2 / Tortoise / Bark** | Top open-model expressiveness or specific styles | Slower / heavier / less of a unified toolkit (Bark also non-deterministic) |
| **ElevenLabs** | Best-in-class naturalness + cloning, zero setup | Paid per-character, cloud-only, data leaves your box |
| **Azure / Google / Amazon Polly TTS** | Managed scale, SSML, SLAs, huge voice catalog | Per-use cost, vendor lock-in, network dependency |
| **espeak-ng / classic formant TTS** | Ultra-light, fully offline, deterministic, accessibility | Robotic, clearly synthetic — fine for screen readers, not narration |

**Rule of thumb:** reach for Coqui when you want **open, self-hosted** speech and especially
**voice cloning** without a cloud bill — it's the most complete open toolkit and XTTS-v2 is
the strongest open zero-shot cloner. Drop to **Piper** when footprint/latency dominate, and
go **cloud (ElevenLabs/Azure/Google)** when you need the highest naturalness with the least
ops and are willing to pay and send audio off-box.

## 8. Resources

- **Coqui TTS docs** — API, model zoo, training guides: https://docs.coqui.ai/en/latest/
- **Community fork (GitHub, maintained)** — install + issues: https://github.com/idiap/coqui-ai-TTS
- **Original repo (archived, still the canonical reference)**: https://github.com/coqui-ai/TTS
- **XTTS-v2 model card** — cloning usage, languages, license: https://huggingface.co/coqui/XTTS-v2
- **VITS paper** — the end-to-end architecture behind much of the zoo: https://arxiv.org/abs/2106.06103
- **HiFi-GAN paper** — the default neural vocoder: https://arxiv.org/abs/2010.05646

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
PAD, BLANK, EOS = "<pad>", "<blk>", "<eos>"


def normalize(text):
    ...


def build_vocab():
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE